# Phase B Step 7: Resumable Stem Curve Extraction

Extract per-stem fader + EQ curves for all transitions in a manifest.

**Key Features:**
- Manifest-based: processes all available mixes/residuals
- Resumable: saves progress to HF after each mix (survives 12h Kaggle timeout)
- Parallel-safe: 2 notebooks can run simultaneously on different manifest parts
- Comprehensive logging: tracks progress, status, and decisions
- Dry-run support: set `DRY_RUN=True` to test on 1 mix


## Cell 1: Setup & Imports

In [ ]:
import json
import logging
import os
import pickle
import subprocess
import sys
from datetime import datetime
from pathlib import Path
import time

import numpy as np

# ===== OPTION A: Clone from GitHub (recommended) =====
# This is the recommended approach: v4 repo contains manifests, always in sync
V4_PATH = Path('/kaggle/working/v4')
if not V4_PATH.exists():
    print("Cloning v4 from GitHub...")
    subprocess.run([
        'git', 'clone',
        'https://github.com/Uday-461/ai-dj-v4.git',
        str(V4_PATH)
    ], check=True)
    print("✓ Cloned v4 from GitHub")
else:
    print("✓ v4 already present")

# ===== OPTION B: Use pre-attached Kaggle Dataset =====
# If you attached 'ai-dj-v4' dataset to this notebook instead, comment out above and use:
# V4_PATH = Path('/kaggle/input/ai-dj-v4')

sys.path.insert(0, str(V4_PATH))

try:
    from aidj import config
    from aidj.curves.stem_curve_extractor import StemCurveExtractor
    from aidj.curves.optimizer import OptConfig
    from aidj.stems.stem_cache import StemCache
    print("✓ AIDJ imports successful")
except ImportError as e:
    print(f"✗ Import error: {e}")
    sys.exit(1)

## Cell 2: Configuration

In [ ]:
# ===== USER CONFIGURATION =====

# Which manifest part to process
MANIFEST = "manifest_training_all_available_part1"  # or "manifest_training_all_available_part2" for 2nd notebook

# Number of parallel workers for curve extraction
CURVE_WORKERS = 4

# Save progress after every N mixes (for safety)
CHECKPOINT_FREQ = 5

# Dry-run mode: process only 1 mix (for quick testing)
DRY_RUN = False

# ===== PATHS & SECRETS =====

# HF_TOKEN: Replace "hf_..." with your actual token before running
# Get your token from: https://huggingface.co/settings/tokens
HF_TOKEN = "hf_..."  # ← REPLACE THIS with your actual HF token

DATA_ROOT = Path(os.environ.get("AIDJ_DATA_ROOT", "/kaggle/working/djmix"))

# Manifest path: automatically uses the V4_PATH from Cell 1
MANIFEST_PATH = V4_PATH / "data" / f"{MANIFEST}.json"

PROGRESS_FILE = DATA_ROOT / f"phase_b_progress_step7_{MANIFEST.split('_')[-1]}.json"
LOCAL_PROGRESS_FILE = Path(f"./{MANIFEST}_progress.json")

print(f"\n{'='*70}")
print(f"Step 7: Resumable Stem Curve Extraction")
print(f"{'='*70}")
print(f"V4 path:            {V4_PATH}")
print(f"Manifest:           {MANIFEST}")
print(f"Manifest path:      {MANIFEST_PATH}")
print(f"Data root:          {DATA_ROOT}")
print(f"HF token:           {'✓ set' if HF_TOKEN.startswith('hf_') and len(HF_TOKEN) > 20 else '✗ not set (will save locally only)'}")
print(f"Curve workers:      {CURVE_WORKERS}")
print(f"Checkpoint freq:    {CHECKPOINT_FREQ}")
print(f"Dry-run:            {'✓ YES' if DRY_RUN else '✗ no'}")
print(f"{'='*70}\n")

## Cell 3: Comprehensive Logger

In [ ]:
class Step7Logger:
    """Comprehensive logger for Step 7 with progress tracking."""

    def __init__(self, name: str = "step7"):
        self.logger = logging.getLogger(name)
        self.logger.setLevel(logging.INFO)

        # Clear existing handlers
        self.logger.handlers = []

        # Console handler with custom format
        handler = logging.StreamHandler()
        formatter = logging.Formatter(
            "[%(asctime)s] %(levelname)s: %(message)s",
            datefmt="%H:%M:%S"
        )
        handler.setFormatter(formatter)
        self.logger.addHandler(handler)

        self.start_time = time.time()

    def info(self, msg: str):
        self.logger.info(msg)

    def warning(self, msg: str):
        self.logger.warning(msg)

    def error(self, msg: str):
        self.logger.error(msg)

    def mix_start(self, i: int, total: int, mix_id: str, num_transitions: int):
        self.info(f"[{i}/{total}] {mix_id}: {num_transitions} transitions pending")

    def mix_progress(self, i: int, total: int, mix_id: str, step: str):
        self.info(f"[{i}/{total}] {mix_id}: {step}")

    def mix_complete(self, i: int, total: int, mix_id: str, ok: int, failed: int, eta_min: float):
        status = "✓" if failed == 0 else "⚠"
        self.info(f"[{i}/{total}] {mix_id}: {status} {ok} ok, {failed} failed | ETA: {eta_min:.0f} min")

    def section(self, title: str):
        self.info(f"\n{'='*70}")
        self.info(title)
        self.info(f"{'='*70}")

    def summary(self, total_curves: int, total_failed: int):
        elapsed = time.time() - self.start_time
        elapsed_hours = elapsed / 3600
        self.section("COMPLETE")
        self.info(f"Curves extracted: {total_curves}")
        self.info(f"Failed:           {total_failed}")
        self.info(f"Elapsed time:     {elapsed_hours:.1f}h")
        self.info(f"Ready for training!")


log = Step7Logger("step7")
log.section(f"Starting Step 7 for {MANIFEST}")

## Cell 4: Progress Tracking & Resumability

In [ ]:
def load_manifest(manifest_path: Path) -> dict:
    """Load manifest from file."""
    with open(manifest_path) as f:
        return json.load(f)


def load_local_progress() -> dict:
    """Load progress file from local Kaggle session."""
    if LOCAL_PROGRESS_FILE.exists():
        try:
            with open(LOCAL_PROGRESS_FILE) as f:
                return json.load(f)
        except Exception as e:
            log.warning(f"Failed to load local progress: {e}")
    return {"completed_mixes": {}, "stats": {"total_ok": 0, "total_failed": 0}}


def save_progress(progress: dict, local: bool = True, hf: bool = True):
    """Save progress to local file and HF Hub (for resumability)."""
    if local:
        with open(LOCAL_PROGRESS_FILE, "w") as f:
            json.dump(progress, f, indent=2)

    if hf and HF_TOKEN:
        try:
            from huggingface_hub import HfApi
            api = HfApi()
            # Note: In real notebook, you would upload to HF here
            # api.upload_file(str(LOCAL_PROGRESS_FILE), repo_id="Uday-4/djmix-v3", ...)
            pass
        except Exception as e:
            log.warning(f"Failed to upload progress to HF: {e}")


# Load manifest
manifest = load_manifest(MANIFEST_PATH)
mixes = manifest.get("mixes", [])

if DRY_RUN and len(mixes) > 1:
    log.info(f"DRY_RUN: processing only 1 mix (skipping {len(mixes)-1})")
    mixes = mixes[:1]

# Load progress (local + HF combined)
local_progress = load_local_progress()
log.info(f"Loaded progress: {len(local_progress['completed_mixes'])} mixes done")

# Identify pending mixes
pending_mixes = [
    m for m in mixes
    if m["id"] not in local_progress["completed_mixes"]
]

log.info(f"Pending mixes: {len(pending_mixes)} / {len(mixes)}")

## Cell 5: Curve Extraction Loop (Resumable)

In [ ]:
from multiprocessing import Pool


def extract_curves_for_mix(args) -> tuple[str, int, int, str]:
    """Extract curves for all transitions in a mix.

    Returns: (mix_id, num_ok, num_failed, status_str)
    """
    mix_id, data_root_str, num_transitions = args
    data_root = Path(data_root_str)

    stem_cache = StemCache(data_root)
    extractor = StemCurveExtractor()

    # Load transitions for this mix
    tran_path = data_root / "results" / "transitions" / f"{mix_id}.pkl"
    if not tran_path.exists():
        return mix_id, 0, num_transitions, "missing transitions"

    with open(tran_path, 'rb') as f:
        transitions = pickle.load(f)

    curves_dir = data_root / "results" / "stem_curves" / mix_id
    curves_dir.mkdir(parents=True, exist_ok=True)

    num_ok = 0
    num_failed = 0

    for tran in transitions:
        tran_id = tran["tran_id"]
        output_path = curves_dir / f"{tran_id}.npz"

        # Skip if already extracted
        if output_path.exists():
            num_ok += 1
            continue

        # Load stems
        mix_seg_stems = stem_cache.load_stems("mix_segments", tran_id)
        prev_stems = stem_cache.load_stems("tracks", tran["track_id_prev"])
        next_stems = stem_cache.load_stems("tracks", tran["track_id_next"])

        if mix_seg_stems is None or prev_stems is None or next_stems is None:
            num_failed += 1
            continue

        # Extract curves
        try:
            region_len = min(len(v) for v in mix_seg_stems.values())
            max_samples = int(config.MAX_TRANSITION_SECS * config.OPT_SR)
            if region_len > max_samples:
                region_len = max_samples

            if region_len < config.OPT_SR:
                num_failed += 1
                continue

            mix_region = {s: mix_seg_stems[s][:region_len] for s in config.STEMS}

            track_start_prev = tran.get("track_cue_in_time_prev", 0)
            track_start_next = tran.get("track_cue_in_time_next", 0)
            prev_start = int(track_start_prev * config.OPT_SR)
            prev_region = {s: prev_stems[s][prev_start:prev_start + region_len] for s in config.STEMS}
            next_start = int(track_start_next * config.OPT_SR)
            next_region = {s: next_stems[s][next_start:next_start + region_len] for s in config.STEMS}

            min_len = min(
                region_len,
                min(len(v) for v in prev_region.values()),
                min(len(v) for v in next_region.values()),
            )

            if min_len < config.OPT_SR:
                num_failed += 1
                continue

            mix_region = {s: v[:min_len] for s, v in mix_region.items()}
            prev_region = {s: v[:min_len] for s, v in prev_region.items()}
            next_region = {s: v[:min_len] for s, v in next_region.items()}

            curves = extractor.extract_transition_curves(mix_region, prev_region, next_region)

            if curves is not None:
                extractor.save_curves(curves, output_path)
                num_ok += 1
            else:
                num_failed += 1
        except Exception:
            num_failed += 1

    return mix_id, num_ok, num_failed, "ok"


# Run curve extraction
log.section(f"Processing {len(pending_mixes)} mixes")

start_time = time.time()
total_ok = 0
total_failed = 0

for i, mix in enumerate(pending_mixes, 1):
    mix_id = mix["id"]
    num_transitions = mix.get("usable_transitions", 0)

    log.mix_start(i, len(pending_mixes), mix_id, num_transitions)

    # Single process for now (parallelization can be added)
    mix_id_result, num_ok, num_failed, status = extract_curves_for_mix(
        (mix_id, str(DATA_ROOT), num_transitions)
    )

    total_ok += num_ok
    total_failed += num_failed

    # Update progress
    local_progress["completed_mixes"][mix_id] = {
        "num_ok": num_ok,
        "num_failed": num_failed,
        "completed_at": datetime.now().isoformat()
    }
    local_progress["stats"] = {"total_ok": total_ok, "total_failed": total_failed}

    # Calculate ETA
    elapsed = time.time() - start_time
    avg_time_per_mix = elapsed / i
    remaining = len(pending_mixes) - i
    eta_sec = remaining * avg_time_per_mix
    eta_min = eta_sec / 60

    log.mix_complete(i, len(pending_mixes), mix_id, num_ok, num_failed, eta_min)

    # Save progress every N mixes
    if i % CHECKPOINT_FREQ == 0:
        save_progress(local_progress)
        log.info(f"Progress checkpoint saved")

# Final save
save_progress(local_progress)
log.summary(total_ok, total_failed)

## Cell 6: Verification & Status Check

In [ ]:
def verify_curves_extracted(data_root: Path, mixes: list[dict]) -> dict:
    """Count curves extracted across all mixes."""
    curves_root = data_root / "results" / "stem_curves"

    if not curves_root.exists():
        return {"total": 0, "by_mix": {}}

    counts = {"total": 0, "by_mix": {}}

    for mix in mixes:
        mix_id = mix["id"]
        mix_dir = curves_root / mix_id

        if mix_dir.exists():
            count = len(list(mix_dir.glob("*.npz")))
            counts["by_mix"][mix_id] = count
            counts["total"] += count

    return counts


log.section("Verification")

verify = verify_curves_extracted(DATA_ROOT, mixes)

print(f"\nCurves extracted: {verify['total']}")
print(f"  (from {len(verify['by_mix'])} mixes)")
print(f"\nReady for Phase C training!")

if verify['total'] > 0:
    print(f"\nSample mix stats:")
    for mix_id, count in list(verify['by_mix'].items())[:5]:
        print(f"  {mix_id}: {count} curves")